In [117]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

deliveries = pd.read_csv(RAW_DIR / "deliveries.csv")
matches = pd.read_csv(RAW_DIR / "matches.csv")
points_table = pd.read_csv(RAW_DIR / "points_table.csv")

print("All datasets loaded successfully!")


All datasets loaded successfully!


In [118]:
print("Deliveries:", deliveries.shape)
print("Matches:", matches.shape)
print("Points table:", points_table.shape)

Deliveries: (26119, 22)
Matches: (48, 18)
Points table: (10, 11)


In [119]:
baseline = {
    "deliveries_rows": len(deliveries),
    "matches_rows": len(matches),
    "points_table_rows": len(points_table)
}
baseline

{'deliveries_rows': 26119, 'matches_rows': 48, 'points_table_rows': 10}

In [120]:
print("Duplicate delivery rows:", deliveries.duplicated().sum())

duplicate_rows = deliveries[deliveries.duplicated(keep=False)].sort_values(deliveries.columns.tolist())
display(duplicate_rows)

Duplicate delivery rows: 1


,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
10383,21,2023/24,2023-10-22,"Himachal Pradesh Cricket Association Stadium, ...",1,9.1,New Zealand,India,DJ Mitchell,R Ravindra,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10392,21,2023/24,2023-10-22,"Himachal Pradesh Cricket Association Stadium, ...",1,9.1,New Zealand,India,DJ Mitchell,R Ravindra,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [121]:

#  Exact duplicate rows remove karo
deliveries = deliveries.drop_duplicates()

print("Deliveries after duplication:", deliveries.shape)
print("Remaining duplicates:", deliveries.duplicated().sum())

Deliveries after duplication: (26118, 22)
Remaining duplicates: 0


In [122]:
# column names and text values clean 
def clean_text_columns(df):
    df = df.copy()
    
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )
    return df

deliveries = clean_text_columns(deliveries)
matches = clean_text_columns(matches)
points_table = clean_text_columns(points_table)

print(deliveries.columns.tolist())
print(matches.columns.tolist())
print(points_table.columns.tolist())



['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed']
['season', 'team1', 'team2', 'date', 'match_number', 'venue', 'city', 'toss_winner', 'toss_decision', 'player_of_match', 'umpire1', 'umpire2', 'reserve_umpire', 'match_referee', 'winner', 'winner_runs', 'winner_wickets', 'match_type']
['ranking', 'team', 'matches', 'won', 'lost', 'tie', 'no_results', 'points', 'net_run_rate', 'for', 'against']


In [123]:

# Reload the original matches dataset
matches_raw = pd.read_csv(RAW_DIR / "matches.csv")

print("Original date examples:")
display(matches_raw[["date", "team1", "team2"]].head(10))

print("\nOriginal date data type:")
print(matches_raw["date"].dtype)

print("\nOriginal missing dates:")
print(matches_raw["date"].isna().sum())

# Check which dates failed after the current conversion
raw_dates = matches_raw["date"].astype("string").str.strip()

parsed_dates = pd.to_datetime(
    raw_dates,
    format="mixed",
    errors="coerce",
    dayfirst=True
)

print("\nMissing dates with mixed-format parsing:")
print(parsed_dates.isna().sum())

print("\nDates that could not be parsed:")
display(matches_raw.loc[
    parsed_dates.isna(),
    ["date", "team1", "team2"]
])

Original date examples:


,date,team1,team2
0,2023/10/05,England,New Zealand
1,2023/10/06,Pakistan,Netherlands
2,2023/10/07,Afghanistan,Bangladesh
3,2023/10/07,South Africa,Sri Lanka
4,2023/10/08,Australia,India
5,2023/10/09,New Zealand,Netherlands
6,2023/10/10,England,Bangladesh
7,2023/10/10,Sri Lanka,Pakistan
8,2023/10/11,Afghanistan,India
9,2023/10/12,South Africa,Australia



Original date data type:
object

Original missing dates:
0

Missing dates with mixed-format parsing:
0

Dates that could not be parsed:


,date,team1,team2


In [124]:

# Correct date conversion

deliveries["start_date"] = pd.to_datetime(
    deliveries["start_date"],
    errors="coerce"
)

matches["date"] = pd.to_datetime(
    matches["date"],
    format="%Y/%m/%d",
    errors="coerce"
)

print("Missing delivery dates:", deliveries["start_date"].isna().sum())
print("Missing match dates:", matches["date"].isna().sum())

display(matches[["date", "team1", "team2"]].head(10))

Missing delivery dates: 0
Missing match dates: 0


,date,team1,team2
0,2023-10-05,England,New Zealand
1,2023-10-06,Pakistan,Netherlands
2,2023-10-07,Afghanistan,Bangladesh
3,2023-10-07,South Africa,Sri Lanka
4,2023-10-08,Australia,India
5,2023-10-09,New Zealand,Netherlands
6,2023-10-10,England,Bangladesh
7,2023-10-10,Sri Lanka,Pakistan
8,2023-10-11,Afghanistan,India
9,2023-10-12,South Africa,Australia


In [125]:
# Extras columns ko clean
extra_columns = [
    "wides",
    "noballs",
    "byes",
    "legbyes",
    "penalty"
]

print("Missing values before filling:")
display(deliveries[extra_columns].isna().sum())

# Verify that non-missing values are numeric
for col in extra_columns:
    deliveries[col] = pd.to_numeric(
        deliveries[col],
        errors="coerce"
    )

# Missing extra subtypes mean no such extra,
# provided this matches the dataset's conventions.
deliveries[extra_columns] = (
    deliveries[extra_columns].fillna(0)
)

print("Missing values after filling:")
display(deliveries[extra_columns].isna().sum())

Missing values before filling:


wides      25453
noballs    26062
byes       26080
legbyes    25925
penalty    26117
dtype: int64

Missing values after filling:


wides      0
noballs    0
byes       0
legbyes    0
penalty    0
dtype: int64

In [126]:

# Delivery numeric columns
delivery_numeric = [
    "match_id", "innings", "ball", "runs_off_bat", "extras"
]

# Match numeric columns
match_numeric = [
    "match_number", "winner_runs", "winner_wickets"
]

# Points table numeric columns
points_numeric = [
    "ranking", "matches", "won", "lost", "tie",
    "no_results", "points", "net_run_rate"
]

def convert_numeric(df, columns, dataset_name):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            print(f"{dataset_name}: Column missing -> {col}")

convert_numeric(deliveries, delivery_numeric, "Deliveries")
convert_numeric(matches, match_numeric, "Matches")
convert_numeric(points_table, points_numeric, "Points table")

print("Delivery numeric columns:")
display(deliveries[[c for c in delivery_numeric if c in deliveries.columns]].describe().T)

print("Match result columns:")
display(matches[[c for c in ["winner_runs", "winner_wickets"] if c in matches.columns]].describe().T)

print("Points table:")
display(points_table.head())

Delivery numeric columns:


,count,mean,std,min,25%,50%,75%,max
match_id,26118.0,24.586645,13.996963,1.0,12.0,24.0,37.0,48.0
innings,26118.0,1.448886,0.497390,1.0,1.0,1.0,2.0,2.0
ball,26118.0,22.648009,13.742183,0.1,10.6,22.1,34.1,49.9
runs_off_bat,26118.0,0.896968,1.399978,0.0,0.0,0.0,1.0,6.0
extras,26118.0,0.046405,0.293629,0.0,0.0,0.0,0.0,5.0


Match result columns:


,count,mean,std,min,25%,50%,75%,max
winner_runs,24.0,125.916667,82.242968,5.0,69.75,101.0,160.00,309.0
winner_wickets,24.0,5.916667,2.019829,1.0,5.00,6.0,7.25,9.0


Points table:


,ranking,team,matches,won,lost,tie,no_results,points,net_run_rate,for,against
0,1,India,9,9,0,0,0,18,2.570,2523/396.2,1708/450.0
1,2,South Africa,9,7,2,0,0,14,1.261,2685/437.5,2158/443.0
2,3,Australia,9,7,2,0,0,14,0.841,2631/426.5,2349/441.2
3,4,New Zealand,9,5,4,0,0,10,0.743,2315/378.0,2279/423.3
4,5,Pakistan,9,4,5,0,0,8,-0.199,2328/406.2,2385/402.2


In [127]:
print("Negative runs off bat:")
print((deliveries["runs_off_bat"] < 0).sum())

print("\nInvalid innings values:")
print((~deliveries["innings"].isin([1, 2])).sum())

print("\nMissing match IDs:")
print(deliveries["match_id"].isna().sum())

print("\nUnique delivery match IDs:")
print(deliveries["match_id"].nunique())

print("\nUnique match records:")
print(matches["match_number"].nunique())

print("\nMissing match winners:")
print(matches["winner"].isna().sum())

print("\nPoints table matches consistency:")
display(
    points_table[
        ["team", "matches", "won", "lost", "tie", "no_results", "points"]
    ]
)

Negative runs off bat:
0

Invalid innings values:
0

Missing match IDs:
0

Unique delivery match IDs:
48

Unique match records:
48

Missing match winners:
0

Points table matches consistency:


,team,matches,won,lost,tie,no_results,points
0,India,9,9,0,0,0,18
1,South Africa,9,7,2,0,0,14
2,Australia,9,7,2,0,0,14
3,New Zealand,9,5,4,0,0,10
4,Pakistan,9,4,5,0,0,8
5,Afghanistan,9,4,5,0,0,8
6,England,9,3,6,0,0,6
7,Bangladesh,9,2,7,0,0,4
8,Sri Lanka,9,2,7,0,0,4
9,Netherlands,9,2,7,0,0,4


In [128]:
# cleaned datasets save 
deliveries.to_csv(
    PROCESSED_DIR / "deliveries_cleaned.csv",
    index=False
)

matches.to_csv(
    PROCESSED_DIR / "matches_cleaned.csv",
    index=False
)

points_table.to_csv(
    PROCESSED_DIR / "points_table_cleaned.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

print("Saved files:")
for file in sorted(PROCESSED_DIR.glob("*.csv")):
    print(file.name)

All cleaned datasets saved successfully!
Saved files:
deliveries_cleaned.csv
matches_cleaned.csv
points_table_cleaned.csv
